In [1]:
class Neuron:
    def __init__(self, id_name, threshold):
        self.id_name = id_name
        self.threshold = threshold
        self.state = 0  # Current state (fires at time t)
        self.next_state = 0 # State for time t+1
        self.excitatory_inputs = [] # List of presynaptic Neuron objects
        self.inhibitory_inputs = [] # List of presynaptic Neuron objects

    def add_excitatory_input(self, neuron):
        self.excitatory_inputs.append(neuron)

    def add_inhibitory_input(self, neuron):
        self.inhibitory_inputs.append(neuron)

    def calculate_next_state(self):
        """Calculates what the state of this neuron will be at t+1,
           based on the states of its inputs at time t."""
        sum_excitatory = 0
        for neuron in self.excitatory_inputs:
            sum_excitatory += neuron.state # Use current state of input neurons

        is_inhibited = False
        for neuron in self.inhibitory_inputs:
            if neuron.state == 1: # If any inhibitory input is firing
                is_inhibited = True
                break

        if not is_inhibited and sum_excitatory >= self.threshold:
            self.next_state = 1
        else:
            self.next_state = 0

    def update_state(self):
        """Sets the current state to the pre-calculated next state."""
        self.state = self.next_state

    def __repr__(self):
        return f"Neuron({self.id_name}, Thr:{self.threshold}, State:{self.state})"

class Network:
    def __init__(self):
        self.neurons = {} # Dictionary to store neurons by id_name
        self.history = [] # List of dictionaries, each recording all neuron states at a time step

    def add_neuron(self, neuron):
        self.neurons[neuron.id_name] = neuron

    def get_neuron(self, id_name):
        return self.neurons.get(id_name)

    def connect(self, pre_neuron_id, post_neuron_id, type="excitatory"):
        pre_neuron = self.get_neuron(pre_neuron_id)
        post_neuron = self.get_neuron(post_neuron_id)
        if not pre_neuron or not post_neuron:
            print(f"Error: Neuron ID not found for connection {pre_neuron_id} -> {post_neuron_id}")
            return

        if type == "excitatory":
            post_neuron.add_excitatory_input(pre_neuron)
        elif type == "inhibitory":
            post_neuron.add_inhibitory_input(pre_neuron)
        else:
            print(f"Unknown connection type: {type}")

    def step(self, external_inputs=None):
        """
        Perform one time step of the simulation.
        external_inputs: A dictionary {neuron_id: state} to set input neuron states.
        """
        # 1. Apply external inputs (these are the states at time 't')
        if external_inputs:
            for neuron_id, state in external_inputs.items():
                if neuron_id in self.neurons:
                    self.neurons[neuron_id].state = state
                else:
                    print(f"Warning: External input for unknown neuron {neuron_id}")

        # 2. Record current states (at time 't') BEFORE calculating next states
        current_states_at_t = {nid: n.state for nid, n in self.neurons.items()}
        self.history.append(current_states_at_t)

        # 3. Calculate next_state (for time 't+1') for all neurons
        #    based on their inputs' states at time 't'
        for neuron in self.neurons.values():
            # Peripheral afferents (input neurons) usually don't have internal inputs
            # in these simple models, their state is dictated by external_inputs.
            # If an input neuron *also* had internal connections, this logic might need adjustment
            # or a flag to indicate it's purely an input.
            # For now, all neurons calculate based on their defined inputs.
            if external_inputs and neuron.id_name in external_inputs:
                # If it's an input neuron, its next_state is effectively 0
                # unless specified again in the *next* call to step().
                # The paper implies that N(t) means it fires at t.
                # So, its contribution to others is its state at t.
                # Its own state at t+1 will be 0 unless it gets another external input at t+1.
                # To model this, we can set its next_state to 0 if it was an external input,
                # UNLESS it also has internal logic that would make it fire.
                # For simplicity: assume external inputs override and don't persist unless resent.
                # The calculate_next_state will handle if it has internal loops making it persist.
                 pass # Its state was set, now let it calculate its next_state based on its connections

            neuron.calculate_next_state()


        # 4. Update all neuron states to their next_state (transitions from 't' to 't+1')
        for neuron in self.neurons.values():
            neuron.update_state()


    def run(self, time_steps, input_sequence=None):
        """
        Run the simulation for a given number of time steps.
        input_sequence: A list of dictionaries, where each dict is the
                        external_inputs for that time step.
                        e.g., [{N1:1, N2:0}, {N1:0, N2:1}, None, {N1:1}]
        """
        self.history = [] # Clear history
        # Initialize all neuron states to 0
        for neuron in self.neurons.values():
            neuron.state = 0
            neuron.next_state = 0

        for t in range(time_steps):
            current_external_inputs = None
            if input_sequence and t < len(input_sequence):
                current_external_inputs = input_sequence[t]
            self.step(external_inputs=current_external_inputs)
        # Add the final state to history as well
        final_states = {nid: n.state for nid, n in self.neurons.items()}
        self.history.append(final_states)


    def print_history(self):
        if not self.history:
            print("No history to print. Run the simulation first.")
            return

        neuron_ids = sorted(self.neurons.keys())
        header = "Time | " + " | ".join(neuron_ids)
        print(header)
        print("-" * len(header))

        # Print history up to the second to last step (recorded states *before* update)
        # The last entry in history is the state *after* the final update.
        for t, step_states in enumerate(self.history):
            # Ensure all neuron_ids are present in step_states, defaulting to 0 if not
            row_values = [str(step_states.get(nid, 0)) for nid in neuron_ids]
            row = f"{t:4} | " + " | ".join(row_values)
            print(row)

# --- Example: Implementing Figure 1a (Delay/Identity) ---
# N2(t) .≡. N1(t-1)
# This means if N1 fires at t-1, N2 fires at t.
# So, if we apply N1 at time=0, N2 should fire at time=1.

print("--- Figure 1a: N2(t) .≡. N1(t-1) ---")
net_1a = Network()
net_1a.add_neuron(Neuron("N1", threshold=1)) # Input neuron, threshold effectively unused if externally driven
net_1a.add_neuron(Neuron("N2", threshold=1))
net_1a.connect("N1", "N2", type="excitatory")

# Input sequence: N1 fires at t=0, then t=2. N1 off at t=1, t=3.
inputs_1a = [
    {"N1": 1}, # t=0
    {"N1": 0}, # t=1
    {"N1": 1}, # t=2
    {"N1": 0}, # t=3
    None        # t=4 (no inputs, observe decay or persistence)
]
net_1a.run(time_steps=5, input_sequence=inputs_1a)
net_1a.print_history()

# --- Example: Implementing Figure 1c (AND gate) ---
# N3(t) .≡. N1(t-1) . N2(t-1)
# N3 fires at 't' if N1 AND N2 fire at 't-1'.
# Threshold of N3 must be 2.
print("\n--- Figure 1c: AND Gate N3(t) .≡. N1(t-1) . N2(t-1) ---")
net_1c = Network()
net_1c.add_neuron(Neuron("N1", threshold=1)) # Input
net_1c.add_neuron(Neuron("N2", threshold=1)) # Input
net_1c.add_neuron(Neuron("N3", threshold=2)) # Output
net_1c.connect("N1", "N3", type="excitatory")
net_1c.connect("N2", "N3", type="excitatory")

# Input sequence for AND: (0,0), (0,1), (1,0), (1,1)
inputs_1c = [
    {"N1": 0, "N2": 0}, # t=0
    {"N1": 0, "N2": 1}, # t=1
    {"N1": 1, "N2": 0}, # t=2
    {"N1": 1, "N2": 1}, # t=3
    None                # t=4
]
net_1c.run(time_steps=5, input_sequence=inputs_1c)
net_1c.print_history()

# --- Example: Implementing Figure 1b (OR gate) ---
# N3(t) .≡. N1(t-1) ∨ N2(t-1)
# N3 fires at 't' if N1 OR N2 (or both) fire at 't-1'.
# Threshold of N3 must be 1.
print("\n--- Figure 1b: OR Gate N3(t) .≡. N1(t-1) ∨ N2(t-1) ---")
net_1b = Network()
net_1b.add_neuron(Neuron("N1", threshold=1))
net_1b.add_neuron(Neuron("N2", threshold=1))
net_1b.add_neuron(Neuron("N3", threshold=1)) # Threshold is 1 for OR
net_1b.connect("N1", "N3", type="excitatory")
net_1b.connect("N2", "N3", type="excitatory")

# Input sequence for OR: (0,0), (0,1), (1,0), (1,1)
inputs_1b = [
    {"N1": 0, "N2": 0}, # t=0
    {"N1": 0, "N2": 1}, # t=1
    {"N1": 1, "N2": 0}, # t=2
    {"N1": 1, "N2": 1}, # t=3
    None                # t=4
]
net_1b.run(time_steps=5, input_sequence=inputs_1b)
net_1b.print_history()


# --- Example: Implementing Figure 1d (N1 AND NOT N2) ---
# N3(t) .≡. N1(t-1) . ~N2(t-1)
# N3 fires at 't' if N1 fires at 't-1' AND N2 DOES NOT fire at 't-1'.
# N3 has an excitatory input from N1 (threshold 1) and an inhibitory input from N2.
print("\n--- Figure 1d: N1 AND NOT N2 ---")
net_1d = Network()
net_1d.add_neuron(Neuron("N1", threshold=1)) # Input
net_1d.add_neuron(Neuron("N2", threshold=1)) # Input (will be inhibitory to N3)
net_1d.add_neuron(Neuron("N3", threshold=1)) # Output
net_1d.connect("N1", "N3", type="excitatory")
net_1d.connect("N2", "N3", type="inhibitory") # N2 inhibits N3

# Input sequence: (0,0), (0,1), (1,0), (1,1)
inputs_1d = [
    {"N1": 0, "N2": 0}, # t=0. N3 should be 0 at t=1
    {"N1": 0, "N2": 1}, # t=1. N3 should be 0 at t=2
    {"N1": 1, "N2": 0}, # t=2. N3 should be 1 at t=3
    {"N1": 1, "N2": 1}, # t=3. N3 should be 0 at t=4
    None                # t=4
]
net_1d.run(time_steps=5, input_sequence=inputs_1d)
net_1d.print_history()

--- Figure 1a: N2(t) .≡. N1(t-1) ---
Time | N1 | N2
--------------
   0 | 1 | 0
   1 | 0 | 1
   2 | 1 | 0
   3 | 0 | 1
   4 | 0 | 0
   5 | 0 | 0

--- Figure 1c: AND Gate N3(t) .≡. N1(t-1) . N2(t-1) ---
Time | N1 | N2 | N3
-------------------
   0 | 0 | 0 | 0
   1 | 0 | 1 | 0
   2 | 1 | 0 | 0
   3 | 1 | 1 | 0
   4 | 0 | 0 | 1
   5 | 0 | 0 | 0

--- Figure 1b: OR Gate N3(t) .≡. N1(t-1) ∨ N2(t-1) ---
Time | N1 | N2 | N3
-------------------
   0 | 0 | 0 | 0
   1 | 0 | 1 | 0
   2 | 1 | 0 | 1
   3 | 1 | 1 | 1
   4 | 0 | 0 | 1
   5 | 0 | 0 | 0

--- Figure 1d: N1 AND NOT N2 ---
Time | N1 | N2 | N3
-------------------
   0 | 0 | 0 | 0
   1 | 0 | 1 | 0
   2 | 1 | 0 | 0
   3 | 1 | 1 | 1
   4 | 0 | 0 | 0
   5 | 0 | 0 | 0


In [2]:


print("\n--- Figure 1e: Heat from Transient Cooling ---")
net_1e = Network()

# Input Neurons
net_1e.add_neuron(Neuron("N1", threshold=1)) # Heat receptor
net_1e.add_neuron(Neuron("N2", threshold=1)) # Cold receptor

# Auxiliary Neurons
net_1e.add_neuron(Neuron("MidA", threshold=1)) # MidA(t) = N2(t-1)
net_1e.add_neuron(Neuron("MidB", threshold=1)) # MidB(t) = MidA(t-1) & ~N2(t-1)

# Output Neurons
net_1e.add_neuron(Neuron("N3", threshold=1))   # Heat sensation: N3(t) = N1(t-1) | MidB(t-1)
net_1e.add_neuron(Neuron("N4", threshold=2))   # Cold sensation: N4(t) = MidA(t-1) & N2(t-1)
                                              # (N4 is directly the AND gate output in Fig 1e)
# Connections
# MidA gets input from N2
net_1e.connect("N2", "MidA", type="excitatory")

# MidB gets input from MidA (excitatory) and N2 (inhibitory)
net_1e.connect("MidA", "MidB", type="excitatory")
net_1e.connect("N2", "MidB", type="inhibitory")

# N3 gets input from N1 (excitatory) and MidB (excitatory)
net_1e.connect("N1", "N3", type="excitatory")
net_1e.connect("MidB", "N3", type="excitatory")

# N4 gets input from MidA (excitatory) and N2 (excitatory)
net_1e.connect("MidA", "N4", type="excitatory")
net_1e.connect("N2", "N4", type="excitatory")


# --- Simulation Scenario 1: Short cold stimulus ---
# N2 fires at t=0, N1 is always off.
print("\nScenario 1: Short Cold Stimulus (N2 fires at t=0)")
inputs_1e_short_cold = [
    {"N1": 0, "N2": 1}, # t=0: Cold stimulus starts
    {"N1": 0, "N2": 0}, # t=1: Cold stimulus ends
    {"N1": 0, "N2": 0}, # t=2
    {"N1": 0, "N2": 0}, # t=3
    {"N1": 0, "N2": 0}, # t=4
    {"N1": 0, "N2": 0}, # t=5
]
net_1e.run(time_steps=6, input_sequence=inputs_1e_short_cold)
net_1e.print_history()
# Expected: N3 (Heat) should fire around t=3. N4 (Cold) might be briefly 0 or 1.

# --- Simulation Scenario 2: Long cold stimulus ---
# N2 fires at t=0, t=1, t=2. N1 is always off.
print("\nScenario 2: Long Cold Stimulus (N2 fires at t=0,1,2)")
inputs_1e_long_cold = [
    {"N1": 0, "N2": 1}, # t=0: Cold stimulus
    {"N1": 0, "N2": 1}, # t=1: Cold stimulus
    {"N1": 0, "N2": 1}, # t=2: Cold stimulus
    {"N1": 0, "N2": 0}, # t=3: Cold stimulus ends
    {"N1": 0, "N2": 0}, # t=4
    {"N1": 0, "N2": 0}, # t=5
    {"N1": 0, "N2": 0}, # t=6
    {"N1": 0, "N2": 0}, # t=7
]
net_1e.run(time_steps=8, input_sequence=inputs_1e_long_cold)
net_1e.print_history()
# Expected: N4 (Cold) should fire during t=2, t=3.
# N3 (Heat) should NOT fire during the cold, but might fire at t=5 (after cold ends).

# --- Simulation Scenario 3: Heat stimulus ---
# N1 fires at t=0. N2 is always off.
print("\nScenario 3: Heat Stimulus (N1 fires at t=0)")
inputs_1e_heat = [
    {"N1": 1, "N2": 0}, # t=0: Heat stimulus
    {"N1": 0, "N2": 0}, # t=1: Heat stimulus ends
    {"N1": 0, "N2": 0}, # t=2
    {"N1": 0, "N2": 0}, # t=3
]
net_1e.run(time_steps=4, input_sequence=inputs_1e_heat)
net_1e.print_history()
# Expected: N3 (Heat) should fire at t=1. N4 (Cold) should remain 0.


--- Figure 1e: Heat from Transient Cooling ---

Scenario 1: Short Cold Stimulus (N2 fires at t=0)
Time | MidA | MidB | N1 | N2 | N3 | N4
--------------------------------------
   0 | 0 | 0 | 0 | 1 | 0 | 0
   1 | 1 | 0 | 0 | 0 | 0 | 0
   2 | 0 | 1 | 0 | 0 | 0 | 0
   3 | 0 | 0 | 0 | 0 | 1 | 0
   4 | 0 | 0 | 0 | 0 | 0 | 0
   5 | 0 | 0 | 0 | 0 | 0 | 0
   6 | 0 | 0 | 0 | 0 | 0 | 0

Scenario 2: Long Cold Stimulus (N2 fires at t=0,1,2)
Time | MidA | MidB | N1 | N2 | N3 | N4
--------------------------------------
   0 | 0 | 0 | 0 | 1 | 0 | 0
   1 | 1 | 0 | 0 | 1 | 0 | 0
   2 | 1 | 0 | 0 | 1 | 0 | 1
   3 | 1 | 0 | 0 | 0 | 0 | 1
   4 | 0 | 1 | 0 | 0 | 0 | 0
   5 | 0 | 0 | 0 | 0 | 1 | 0
   6 | 0 | 0 | 0 | 0 | 0 | 0
   7 | 0 | 0 | 0 | 0 | 0 | 0
   8 | 0 | 0 | 0 | 0 | 0 | 0

Scenario 3: Heat Stimulus (N1 fires at t=0)
Time | MidA | MidB | N1 | N2 | N3 | N4
--------------------------------------
   0 | 0 | 0 | 1 | 0 | 0 | 0
   1 | 0 | 0 | 0 | 0 | 1 | 0
   2 | 0 | 0 | 0 | 0 | 0 | 0
   3 | 0 | 0 | 0 

In [3]:
# (Assuming the Neuron and Network classes from the previous response are defined)

print("\n--- Theorem 4: Relative vs. Absolute Inhibition (Inspired by Fig 1f) ---")

# We want to mimic a neuron Target_Rel with:
# Inputs: E1, E2 (excitatory), I_Rel (relative inhibitor)
# Base Threshold: 1
# If I_Rel=0: Fires if E1 or E2 is 1 (sum >= 1)
# If I_Rel=1: Effective Threshold becomes 2. Fires if E1 AND E2 are 1 (sum >= 2)

# Network using absolute inhibition to mimic Target_Rel
net_Th4 = Network()

# Input neurons for our test
net_Th4.add_neuron(Neuron("E1", 1))
net_Th4.add_neuron(Neuron("E2", 1))
net_Th4.add_neuron(Neuron("I_Rel", 1)) # This will control which path is active

# Output neuron (this is our N_Mimic that behaves like Target_Rel)
net_Th4.add_neuron(Neuron("N_Mimic", 1))

# Path 1: Active when I_Rel is OFF (implements OR logic: E1 | E2)
# Needs: (E1 | E2) & ~I_Rel
net_Th4.add_neuron(Neuron("OR_E1_E2", 1)) # Threshold 1 for OR
net_Th4.connect("E1", "OR_E1_E2")
net_Th4.connect("E2", "OR_E1_E2")

net_Th4.add_neuron(Neuron("Path1_Active", 1)) # Threshold 1
net_Th4.connect("OR_E1_E2", "Path1_Active")
net_Th4.connect("I_Rel", "Path1_Active", type="inhibitory") # If I_Rel is ON, Path1 is OFF

# Path 2: Active when I_Rel is ON (implements AND logic: E1 & E2)
# Needs: (E1 & E2) & I_Rel
net_Th4.add_neuron(Neuron("AND_E1_E2", 2)) # Threshold 2 for AND
net_Th4.connect("E1", "AND_E1_E2")
net_Th4.connect("E2", "AND_E1_E2")

net_Th4.add_neuron(Neuron("Path2_Gate", 1)) # Auxiliary to ensure I_Rel is on for this path
net_Th4.connect("I_Rel", "Path2_Gate") # Path2_Gate is ON if I_Rel is ON

net_Th4.add_neuron(Neuron("Path2_Active", 2)) # Threshold 2: needs AND_E1_E2 AND Path2_Gate
net_Th4.connect("AND_E1_E2", "Path2_Active")
net_Th4.connect("Path2_Gate", "Path2_Active")


# N_Mimic fires if Path1_Active OR Path2_Active fires
net_Th4.connect("Path1_Active", "N_Mimic")
net_Th4.connect("Path2_Active", "N_Mimic")


# Test Scenarios
# (E1, E2, I_Rel) -> N_Mimic (expected)
# If I_Rel = 0:
# (0,0,0) -> 0
# (1,0,0) -> 1
# (0,1,0) -> 1
# (1,1,0) -> 1
# If I_Rel = 1:
# (0,0,1) -> 0
# (1,0,1) -> 0
# (0,1,1) -> 0
# (1,1,1) -> 1

inputs_Th4 = [
    # I_Rel = 0 (OR logic for E1, E2)
    {"E1": 0, "E2": 0, "I_Rel": 0}, # t=0. N_Mimic @ t=2 should be 0
    {"E1": 1, "E2": 0, "I_Rel": 0}, # t=1. N_Mimic @ t=3 should be 1
    {"E1": 0, "E2": 1, "I_Rel": 0}, # t=2. N_Mimic @ t=4 should be 1
    {"E1": 1, "E2": 1, "I_Rel": 0}, # t=3. N_Mimic @ t=5 should be 1
    # I_Rel = 1 (AND logic for E1, E2)
    {"E1": 0, "E2": 0, "I_Rel": 1}, # t=4. N_Mimic @ t=6 should be 0
    {"E1": 1, "E2": 0, "I_Rel": 1}, # t=5. N_Mimic @ t=7 should be 0
    {"E1": 0, "E2": 1, "I_Rel": 1}, # t=6. N_Mimic @ t=8 should be 0
    {"E1": 1, "E2": 1, "I_Rel": 1}, # t=7. N_Mimic @ t=9 should be 1
    None, # t=8
    None, # t=9
]

# The intermediate neurons add delays. OR_E1_E2 fires at t+1. Path1_Active fires at t+2.
# N_Mimic will fire at t+3 relative to the inputs E1,E2,I_Rel.
# So we need to run for enough steps.
# Time: Input -> Mid1(t+1) -> Mid2(t+2) -> Output(t+3)
net_Th4.run(time_steps=11, input_sequence=inputs_Th4)
net_Th4.print_history()


--- Theorem 4: Relative vs. Absolute Inhibition (Inspired by Fig 1f) ---
Time | AND_E1_E2 | E1 | E2 | I_Rel | N_Mimic | OR_E1_E2 | Path1_Active | Path2_Active | Path2_Gate
--------------------------------------------------------------------------------------------------
   0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0
   1 | 0 | 1 | 0 | 0 | 0 | 0 | 0 | 0 | 0
   2 | 0 | 0 | 1 | 0 | 0 | 1 | 0 | 0 | 0
   3 | 0 | 1 | 1 | 0 | 0 | 1 | 1 | 0 | 0
   4 | 1 | 0 | 0 | 1 | 1 | 1 | 1 | 0 | 0
   5 | 0 | 1 | 0 | 1 | 1 | 0 | 0 | 0 | 1
   6 | 0 | 0 | 1 | 1 | 0 | 1 | 0 | 0 | 1
   7 | 0 | 1 | 1 | 1 | 0 | 1 | 0 | 0 | 1
   8 | 1 | 0 | 0 | 0 | 0 | 1 | 0 | 0 | 1
   9 | 0 | 0 | 0 | 0 | 0 | 0 | 1 | 1 | 0
  10 | 0 | 0 | 0 | 0 | 1 | 0 | 0 | 0 | 0
  11 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0


In [9]:
class Neuron:
    def __init__(self, id_name, threshold):
        self.id_name = id_name
        self.threshold = threshold
        self.state = 0  # Current state (fires at time t)
        self.next_state = 0 # State for time t+1
        self.excitatory_inputs = [] # List of presynaptic Neuron objects
        self.inhibitory_inputs = [] # List of presynaptic Neuron objects

    def add_excitatory_input(self, neuron):
        self.excitatory_inputs.append(neuron)

    def add_inhibitory_input(self, neuron):
        self.inhibitory_inputs.append(neuron)

    def calculate_next_state(self):
        """Calculates what the state of this neuron will be at t+1,
           based on the states of its inputs at time t."""
        sum_excitatory = 0
        for neuron in self.excitatory_inputs:
            sum_excitatory += neuron.state # Use current state of input neurons

        is_inhibited = False
        for neuron in self.inhibitory_inputs:
            if neuron.state == 1: # If any inhibitory input is firing
                is_inhibited = True
                break

        if not is_inhibited and sum_excitatory >= self.threshold:
            self.next_state = 1
        else:
            self.next_state = 0

    def update_state(self):
        """Sets the current state to the pre-calculated next state."""
        self.state = self.next_state

    def __repr__(self):
        return f"Neuron({self.id_name}, Thr:{self.threshold}, State:{self.state})"

class Network:
    def __init__(self):
        self.neurons = {} # Dictionary to store neurons by id_name
        self.history = [] # List of dictionaries, each recording all neuron states at a time step

    def add_neuron(self, neuron):
        self.neurons[neuron.id_name] = neuron

    def get_neuron(self, id_name):
        return self.neurons.get(id_name)

    def connect(self, pre_neuron_id, post_neuron_id, type="excitatory"):
        pre_neuron = self.get_neuron(pre_neuron_id)
        post_neuron = self.get_neuron(post_neuron_id)
        if not pre_neuron or not post_neuron:
            print(f"Error: Neuron ID not found for connection {pre_neuron_id} -> {post_neuron_id}")
            return

        if type == "excitatory":
            post_neuron.add_excitatory_input(pre_neuron)
        elif type == "inhibitory":
            post_neuron.add_inhibitory_input(pre_neuron)
        else:
            print(f"Unknown connection type: {type}")

    def step(self, external_inputs=None):
        """
        Perform one time step of the simulation.
        external_inputs: A dictionary {neuron_id: state} to set input neuron states.
        """
        # 1. Apply external inputs (these are the states at time 't')
        if external_inputs:
            for neuron_id, state in external_inputs.items():
                if neuron_id in self.neurons:
                    self.neurons[neuron_id].state = state
                    # Ensure next_state is also reset if it's an external input,
                    # unless it has internal logic to keep it firing.
                    # For pure input neurons, their next_state would naturally be 0
                    # if they have no excitatory inputs themselves.
                    # self.neurons[neuron_id].next_state = state # Or 0 if it should not persist
                else:
                    print(f"Warning: External input for unknown neuron {neuron_id}")

        # 2. Record current states (at time 't') BEFORE calculating next states
        current_states_at_t = {nid: n.state for nid, n in self.neurons.items()}
        self.history.append(current_states_at_t)

        # 3. Calculate next_state (for time 't+1') for all neurons
        #    based on their inputs' states at time 't'
        for neuron_id in self.neurons: # Iterate by ID to ensure consistent order if needed
            neuron = self.neurons[neuron_id]
            # If this neuron was externally driven AND has no internal inputs,
            # its next_state would become 0. calculate_next_state handles this.
            neuron.calculate_next_state()


        # 4. Update all neuron states to their next_state (transitions from 't' to 't+1')
        for neuron in self.neurons.values():
            neuron.update_state()


    def run(self, time_steps, input_sequence=None):
        """
        Run the simulation for a given number of time steps.
        input_sequence: A list of dictionaries, where each dict is the
                        external_inputs for that time step.
                        e.g., [{N1:1, N2:0}, {N1:0, N2:1}, None, {N1:1}]
        """
        self.history = [] # Clear history
        # Initialize all neuron states to 0
        for neuron in self.neurons.values():
            neuron.state = 0
            neuron.next_state = 0

        for t in range(time_steps):
            current_external_inputs = None
            if input_sequence and t < len(input_sequence):
                current_external_inputs = input_sequence[t]
            # Important: If an external input is None for a step,
            # input neurons that were on might turn off unless they have recurrent excitation.
            # For this, we need to ensure their state is explicitly set to 0 if not in current_external_inputs
            # OR that calculate_next_state handles this by them having no active inputs.
            # Let's ensure input neurons (those listed in external_inputs)
            # are reset if not specified in the current step's external_inputs.
            if current_external_inputs is not None:
                all_possible_inputs_this_step = current_external_inputs.keys()
            else:
                all_possible_inputs_this_step = []

            # If an input neuron was active in the previous step due to external_inputs,
            # but is NOT specified in the current step's external_inputs,
            # its state should become 0 before step calculation (unless it has other sustaining inputs).
            # This is tricky. The current model assumes an input neuron, if not given
            # an external input, will calculate its state based on its *internal* connections.
            # For pure input neurons (no internal inputs), this means they go to 0.
            # This behavior seems correct for the McCulloch-Pitts model.

            self.step(external_inputs=current_external_inputs)

        # Add the final state to history as well (after the last update)
        final_states = {nid: n.state for nid, n in self.neurons.items()}
        self.history.append(final_states)


    def print_history(self, neuron_ids=None): # Added neuron_ids as an optional argument
        if not self.history:
            print("No history to print. Run the simulation first.")
            return

        if neuron_ids is None: # If no specific IDs are given, use all
            # Filter out neurons that might not have been active or defined in all steps if not careful
            # Best to sort the keys from self.neurons to ensure all defined neurons are considered
            printable_neuron_ids = sorted(self.neurons.keys())
        else: # Otherwise, use the provided list
            printable_neuron_ids = neuron_ids

        header = "Time | " + " | ".join(f"{nid:^7}" for nid in printable_neuron_ids) # Make columns wider
        print(header)
        print("-" * len(header))

        # self.history contains states *at* time t (before next_state calculation for t+1)
        # The last entry is the state *after* the final update.
        for t, step_states in enumerate(self.history):
            row_values = [str(step_states.get(nid, 0)) for nid in printable_neuron_ids]
            row = f"{t:<4} | " + " | ".join(f"{val:^7}" for val in row_values)
            print(row)

# --- Figure 1a (Delay/Identity) ---
print("--- Figure 1a: N2(t) .≡. N1(t-1) ---")
net_1a = Network()
net_1a.add_neuron(Neuron("N1", threshold=1))
net_1a.add_neuron(Neuron("N2", threshold=1))
net_1a.connect("N1", "N2", type="excitatory")
inputs_1a = [{"N1": 1}, {"N1": 0}, {"N1": 1}, {"N1": 0}, None]
net_1a.run(time_steps=5, input_sequence=inputs_1a)
net_1a.print_history(neuron_ids=["N1", "N2"])
# Expected N2 at t=1 is N1 at t=0.
# History row t=0: N1=1, N2=0  (Input applied)
# History row t=1: N1=0, N2=1  (N2 reflects N1(0), N1 input for t=1 applied)

# --- Figure 1c (AND gate) ---
print("\n--- Figure 1c: AND Gate N3(t) .≡. N1(t-1) . N2(t-1) ---")
net_1c = Network()
net_1c.add_neuron(Neuron("N1", threshold=1))
net_1c.add_neuron(Neuron("N2", threshold=1))
net_1c.add_neuron(Neuron("N3", threshold=2))
net_1c.connect("N1", "N3", type="excitatory")
net_1c.connect("N2", "N3", type="excitatory")
inputs_1c = [{"N1": 0, "N2": 0}, {"N1": 0, "N2": 1}, {"N1": 1, "N2": 0}, {"N1": 1, "N2": 1}, None]
net_1c.run(time_steps=5, input_sequence=inputs_1c)
net_1c.print_history(neuron_ids=["N1", "N2", "N3"])
# Expected N3(t=4) = 1 if N1(t=3)=1 and N2(t=3)=1.

# --- Figure 1b (OR gate) ---
print("\n--- Figure 1b: OR Gate N3(t) .≡. N1(t-1) ∨ N2(t-1) ---")
net_1b = Network()
net_1b.add_neuron(Neuron("N1", threshold=1))
net_1b.add_neuron(Neuron("N2", threshold=1))
net_1b.add_neuron(Neuron("N3", threshold=1))
net_1b.connect("N1", "N3", type="excitatory")
net_1b.connect("N2", "N3", type="excitatory")
inputs_1b = [{"N1": 0, "N2": 0}, {"N1": 0, "N2": 1}, {"N1": 1, "N2": 0}, {"N1": 1, "N2": 1}, None]
net_1b.run(time_steps=5, input_sequence=inputs_1b)
net_1b.print_history(neuron_ids=["N1", "N2", "N3"])

# --- Figure 1d (N1 AND NOT N2) ---
print("\n--- Figure 1d: N1 AND NOT N2 ---")
net_1d = Network()
net_1d.add_neuron(Neuron("N1", threshold=1))
net_1d.add_neuron(Neuron("N2", threshold=1))
net_1d.add_neuron(Neuron("N3", threshold=1))
net_1d.connect("N1", "N3", type="excitatory")
net_1d.connect("N2", "N3", type="inhibitory")
inputs_1d = [{"N1": 0, "N2": 0}, {"N1": 0, "N2": 1}, {"N1": 1, "N2": 0}, {"N1": 1, "N2": 1}, None]
net_1d.run(time_steps=5, input_sequence=inputs_1d)
net_1d.print_history(neuron_ids=["N1", "N2", "N3"])
# Expected N3(t=3)=1 for N1(t=2)=1, N2(t=2)=0.

# --- Theorem 5: Extinction M=2 (Parallel inhibitory circuits) ---
# Goal: If N_Mimic fires once (due to a single pulse E_in),
# it should be unable to fire for the next M=2 time steps, even if E_in persisted.
# Circuit T1 (length 1 path for inhibition at t+1): N_Mimic -> I1_End --| N_Mimic
# Circuit T2 (length 2 path for inhibition at t+2): N_Mimic -> I2_Mid -> I2_End --| N_Mimic
net_Th5_M2_parallel = Network()
net_Th5_M2_parallel.add_neuron(Neuron("E_in", 1))
net_Th5_M2_parallel.add_neuron(Neuron("N_Mimic", 1))

# Circuit T1
net_Th5_M2_parallel.add_neuron(Neuron("I1_End", 1))
net_Th5_M2_parallel.connect("N_Mimic", "I1_End")
net_Th5_M2_parallel.connect("I1_End", "N_Mimic", type="inhibitory")

# Circuit T2
net_Th5_M2_parallel.add_neuron(Neuron("I2_Mid", 1))
net_Th5_M2_parallel.add_neuron(Neuron("I2_End", 1))
net_Th5_M2_parallel.connect("N_Mimic", "I2_Mid")
net_Th5_M2_parallel.connect("I2_Mid", "I2_End")
net_Th5_M2_parallel.connect("I2_End", "N_Mimic", type="inhibitory")

print("\n--- Theorem 5: Extinction M=2 (Parallel, Single Pulse E_in) ---")
# E_in fires only at t=0. N_Mimic should fire at t=1.
# Then N_Mimic should be off at t=2 (due to I1_End) and t=3 (due to I2_End).
inputs_Th5_M2_parallel_single = [ {"E_in": 1}, None, None, None, None, None ]
net_Th5_M2_parallel.run(time_steps=6, input_sequence=inputs_Th5_M2_parallel_single)
net_Th5_M2_parallel.print_history(neuron_ids=["E_in", "N_Mimic", "I1_End", "I2_Mid", "I2_End"])
# Expected N_Mimic: 0 (t0), 1 (t1), 0 (t2), 0 (t3), 0 (t4), 0 (t5)

print("\n--- Theorem 5: Extinction M=2 (Parallel, Sustained E_in) ---")
# With sustained input, N_Mimic fires at t=1.
# I1_End(2) inhibits N_Mimic(2).
# I2_End(3) inhibits N_Mimic(3).
# What happens with N_Mimic(1) causing I1_End(2) and I2_Mid(2)?
# The previous trace for sustained input was: N_Mimic: 0, 1, 1, 0, 0, 0, 1, ...
# Let's re-verify the trace carefully based on the code's execution order:

# t=0: E_in=1. All others 0.
#      History(0): E_in=1, N_Mimic=0, I1_End=0, I2_Mid=0, I2_End=0
#      Next states for t=1:
#          N_Mimic_n = E_in(0) & ~I1_End(0) & ~I2_End(0) = 1 & ~0 & ~0 = 1
#          I1_End_n  = N_Mimic(0) = 0
#          I2_Mid_n  = N_Mimic(0) = 0
#          I2_End_n  = I2_Mid(0)  = 0
#
# t=1 (States updated): E_in=1 (external), N_Mimic=1, I1_End=0, I2_Mid=0, I2_End=0
#      History(1): E_in=1, N_Mimic=1, I1_End=0, I2_Mid=0, I2_End=0
#      Next states for t=2:
#          I1_End_n  = N_Mimic(1) = 1
#          I2_Mid_n  = N_Mimic(1) = 1
#          I2_End_n  = I2_Mid(1)  = 0
#          N_Mimic_n = E_in(1) & ~I1_End(1) & ~I2_End(1) = 1 & ~0 & ~0 = 1
#
# t=2 (States updated): E_in=1, N_Mimic=1, I1_End=1, I2_Mid=1, I2_End=0
#      History(2): E_in=1, N_Mimic=1, I1_End=1, I2_Mid=1, I2_End=0
#      Next states for t=3:
#          I1_End_n  = N_Mimic(2) = 1
#          I2_Mid_n  = N_Mimic(2) = 1
#          I2_End_n  = I2_Mid(2)  = 1
#          N_Mimic_n = E_in(2) & ~I1_End(2) & ~I2_End(2) = 1 & ~1 & ~0 = 0
#
# t=3 (States updated): E_in=1, N_Mimic=0, I1_End=1, I2_Mid=1, I2_End=1
#      History(3): E_in=1, N_Mimic=0, I1_End=1, I2_Mid=1, I2_End=1
#      Next states for t=4:
#          I1_End_n  = N_Mimic(3) = 0
#          I2_Mid_n  = N_Mimic(3) = 0
#          I2_End_n  = I2_Mid(3)  = 1
#          N_Mimic_n = E_in(3) & ~I1_End(3) & ~I2_End(3) = 1 & ~1 & ~1 = 0
#
# t=4 (States updated): E_in=1, N_Mimic=0, I1_End=0, I2_Mid=0, I2_End=1
#      History(4): E_in=1, N_Mimic=0, I1_End=0, I2_Mid=0, I2_End=1
#      Next states for t=5:
#          I1_End_n  = N_Mimic(4) = 0
#          I2_Mid_n  = N_Mimic(4) = 0
#          I2_End_n  = I2_Mid(4)  = 0
#          N_Mimic_n = E_in(4) & ~I1_End(4) & ~I2_End(4) = 1 & ~0 & ~1 = 0
#
# t=5 (States updated): E_in=1, N_Mimic=0, I1_End=0, I2_Mid=0, I2_End=0
#      History(5): E_in=1, N_Mimic=0, I1_End=0, I2_Mid=0, I2_End=0
#      Next states for t=6:
#          I1_End_n  = N_Mimic(5) = 0
#          I2_Mid_n  = N_Mimic(5) = 0
#          I2_End_n  = I2_Mid(5)  = 0
#          N_Mimic_n = E_in(5) & ~I1_End(5) & ~I2_End(5) = 1 & ~0 & ~0 = 1

# N_Mimic pattern for sustained input: 0 (t0), 1 (t1), 1 (t2), 0 (t3), 0 (t4), 0 (t5), 1 (t6), ...
# It fires at t=1 and t=2.
# It is then off for t=3, t=4, t=5 (3 steps). This is an M=3 refractory period.
# The length of the refractory period is M_path_max + 1, where M_path_max is the
# number of neurons in the longest inhibitory feedback chain (here, 2 neurons: I2_Mid, I2_End).

inputs_Th5_M2_parallel_sustained = [ {"E_in": 1} for _ in range(12) ]
net_Th5_M2_parallel.run(time_steps=12, input_sequence=inputs_Th5_M2_parallel_sustained)
net_Th5_M2_parallel.print_history(neuron_ids=["E_in", "N_Mimic", "I1_End", "I2_Mid", "I2_End"])

--- Figure 1a: N2(t) .≡. N1(t-1) ---
Time |   N1    |   N2   
------------------------
0    |    1    |    0   
1    |    0    |    1   
2    |    1    |    0   
3    |    0    |    1   
4    |    0    |    0   
5    |    0    |    0   

--- Figure 1c: AND Gate N3(t) .≡. N1(t-1) . N2(t-1) ---
Time |   N1    |   N2    |   N3   
----------------------------------
0    |    0    |    0    |    0   
1    |    0    |    1    |    0   
2    |    1    |    0    |    0   
3    |    1    |    1    |    0   
4    |    0    |    0    |    1   
5    |    0    |    0    |    0   

--- Figure 1b: OR Gate N3(t) .≡. N1(t-1) ∨ N2(t-1) ---
Time |   N1    |   N2    |   N3   
----------------------------------
0    |    0    |    0    |    0   
1    |    0    |    1    |    0   
2    |    1    |    0    |    1   
3    |    1    |    1    |    1   
4    |    0    |    0    |    1   
5    |    0    |    0    |    0   

--- Figure 1d: N1 AND NOT N2 ---
Time |   N1    |   N2    |   N3   
---------------------

In [10]:
# Assuming Neuron and Network classes are defined as before

print("\n--- Theorem 6: Temporal Summation by Spatial Summation (Fig 1h) ---")
net_Th6 = Network()

# Neurons
net_Th6.add_neuron(Neuron("N_Input", threshold=1))       # External input source
net_Th6.add_neuron(Neuron("N_DelayBranch", threshold=1)) # c1 in Fig 1h (delay path)
net_Th6.add_neuron(Neuron("N_Target", threshold=2))      # c2 in Fig 1h (summation neuron)

# Connections
# N_Input provides direct input to N_Target
net_Th6.connect("N_Input", "N_Target", type="excitatory")
# N_Input also excites N_DelayBranch
net_Th6.connect("N_Input", "N_DelayBranch", type="excitatory")
# N_DelayBranch provides the delayed input to N_Target
net_Th6.connect("N_DelayBranch", "N_Target", type="excitatory")

# Scenario 1: Single Pulse Input
print("\nScenario 1: Single Pulse to N_Input (N_Target should NOT fire)")
# N_Input fires at t=0 only
inputs_Th6_single = [
    {"N_Input": 1}, # t=0
    None,           # t=1 (N_Input is off)
    None,           # t=2
    None            # t=3
]
net_Th6.run(time_steps=4, input_sequence=inputs_Th6_single)
net_Th6.print_history(neuron_ids=["N_Input", "N_DelayBranch", "N_Target"])
# Expected N_Target remains 0.
# Trace:
# t=0: N_Input=1. N_DelayBranch=0, N_Target=0
#   Next for t=1: N_DelayBranch_n=1 (from N_Input(0)), N_Target_n=0 (sum=N_Input(0)=1 < thr=2)
# t=1: N_Input=0. N_DelayBranch=1, N_Target=0
#   Next for t=2: N_DelayBranch_n=0 (from N_Input(1)=0), N_Target_n=0 (sum=N_DelayBranch(1)=1 < thr=2)
# t=2: N_Input=0. N_DelayBranch=0, N_Target=0
#   Next for t=3: N_DelayBranch_n=0, N_Target_n=0

# Scenario 2: Two Consecutive Pulses Input
print("\nScenario 2: Two Consecutive Pulses to N_Input (N_Target SHOULD fire at t=2)")
# N_Input fires at t=0 and t=1
inputs_Th6_two_pulses = [
    {"N_Input": 1}, # t=0
    {"N_Input": 1}, # t=1
    None,           # t=2 (N_Input is off)
    None            # t=3
]
net_Th6.run(time_steps=4, input_sequence=inputs_Th6_two_pulses)
net_Th6.print_history(neuron_ids=["N_Input", "N_DelayBranch", "N_Target"])
# Expected N_Target fires at t=2.
# Trace:
# t=0: N_Input=1. N_DelayBranch=0, N_Target=0
#   Next for t=1: N_DelayBranch_n=1 (from N_Input(0)), N_Target_n=0 (sum=N_Input(0)=1 < thr=2)
# t=1: N_Input=1 (external). N_DelayBranch=1, N_Target=0
#   Inputs for N_Target for state at t=2: N_Input(1)=1, N_DelayBranch(1)=1. Sum = 2.
#   Next for t=2:
#     N_DelayBranch_n=1 (from N_Input(1)=1)
#     N_Target_n=1 (sum=N_Input(1)+N_DelayBranch(1) = 1+1=2 >= thr=2)
# t=2: N_Input=0 (external). N_DelayBranch=1, N_Target=1
#   Inputs for N_Target for state at t=3: N_Input(2)=0, N_DelayBranch(2)=1. Sum = 1.
#   Next for t=3:
#     N_DelayBranch_n=0 (from N_Input(2)=0)
#     N_Target_n=0 (sum=N_Input(2)+N_DelayBranch(2) = 0+1=1 < thr=2)
# t=3: N_Input=0. N_DelayBranch=0, N_Target=0


--- Theorem 6: Temporal Summation by Spatial Summation (Fig 1h) ---

Scenario 1: Single Pulse to N_Input (N_Target should NOT fire)
Time | N_Input | N_DelayBranch | N_Target
-----------------------------------------
0    |    1    |    0    |    0   
1    |    0    |    1    |    0   
2    |    0    |    0    |    0   
3    |    0    |    0    |    0   
4    |    0    |    0    |    0   

Scenario 2: Two Consecutive Pulses to N_Input (N_Target SHOULD fire at t=2)
Time | N_Input | N_DelayBranch | N_Target
-----------------------------------------
0    |    1    |    0    |    0   
1    |    1    |    1    |    0   
2    |    0    |    1    |    1   
3    |    0    |    0    |    0   
4    |    0    |    0    |    0   


In [11]:
# Assuming Neuron and Network classes are defined as before

print("\n--- Theorem 7: Alterable Synapses by Circles (Fig 1i logic) ---")
net_Th7 = Network()

# Input Neurons
net_Th7.add_neuron(Neuron("N1_Prime", threshold=1))     # Prime input (c1)
net_Th7.add_neuron(Neuron("N2_Pairing", threshold=1)) # Pairing input (c2)

# Latch Mechanism (to represent (Ex)t-1 . N1(x) . N2(x))
net_Th7.add_neuron(Neuron("N_AND_Inputs", threshold=2)) # Detects simultaneous N1 & N2
net_Th7.add_neuron(Neuron("N_Latch", threshold=1))      # The memory latch itself

# Output Mechanism
net_Th7.add_neuron(Neuron("N_AND_PrimeLatch", threshold=2)) # Fires if N1_Prime AND N_Latch
net_Th7.add_neuron(Neuron("N_Target", threshold=1))         # Final output (c3)

# Connections for Latch:
# N1_Prime and N2_Pairing feed into N_AND_Inputs
net_Th7.connect("N1_Prime", "N_AND_Inputs", type="excitatory")
net_Th7.connect("N2_Pairing", "N_AND_Inputs", type="excitatory")

# N_AND_Inputs (if active) excites N_Latch
net_Th7.connect("N_AND_Inputs", "N_Latch", type="excitatory")
# N_Latch has a self-excitatory loop to stay active once fired
net_Th7.connect("N_Latch", "N_Latch", type="excitatory")


# Connections for Target Output:
# N2_Pairing directly excites N_Target
net_Th7.connect("N2_Pairing", "N_Target", type="excitatory")

# N1_Prime and N_Latch feed into N_AND_PrimeLatch
net_Th7.connect("N1_Prime", "N_AND_PrimeLatch", type="excitatory")
net_Th7.connect("N_Latch", "N_AND_PrimeLatch", type="excitatory")

# N_AND_PrimeLatch excites N_Target
net_Th7.connect("N_AND_PrimeLatch", "N_Target", type="excitatory")


# --- Test Scenarios ---
neuron_order_Th7 = ["N1_Prime", "N2_Pairing", "N_AND_Inputs", "N_Latch", "N_AND_PrimeLatch", "N_Target"]

# Scenario 1: N1_Prime alone (before pairing)
print("\nScenario 1: N1_Prime alone (before pairing) - N_Target should NOT fire from N1_Prime")
inputs_s1 = [
    {"N1_Prime": 1, "N2_Pairing": 0}, # t=0
    None,                             # t=1
    None                              # t=2
]
net_Th7.run(time_steps=3, input_sequence=inputs_s1)
net_Th7.print_history(neuron_ids=neuron_order_Th7)
# Expected: N_Latch=0, N_AND_PrimeLatch=0, N_Target=0

# Scenario 2: N2_Pairing alone
print("\nScenario 2: N2_Pairing alone - N_Target SHOULD fire (direct path)")
inputs_s2 = [
    {"N1_Prime": 0, "N2_Pairing": 1}, # t=0
    None,                             # t=1
    None                              # t=2
]
net_Th7.run(time_steps=3, input_sequence=inputs_s2)
net_Th7.print_history(neuron_ids=neuron_order_Th7)
# Expected: N_Target=1 at t=1 (due to N2_Pairing(0))

# Scenario 3: Pairing Event (N1_Prime AND N2_Pairing together)
print("\nScenario 3: Pairing Event (N1_Prime AND N2_Pairing) - Sets Latch")
inputs_s3 = [
    {"N1_Prime": 1, "N2_Pairing": 1}, # t=0: Pairing
    {"N1_Prime": 0, "N2_Pairing": 0}, # t=1: Inputs off, Latch should persist
    None,                             # t=2: Latch should still persist
    None                              # t=3
]
net_Th7.run(time_steps=4, input_sequence=inputs_s3)
net_Th7.print_history(neuron_ids=neuron_order_Th7)
# Expected:
# t=0: N1=1, N2=1
# t=1: N_AND_Inputs=1, N_Target=1 (from N2). N_Latch=0 initially
# t=2: N_Latch=1 (from N_AND_Inputs(1)), N_Target=0 (inputs are off)
# t=3: N_Latch=1 (self-sustaining), N_Target=0

# Scenario 4: N1_Prime alone (AFTER pairing has occurred and Latch is set)
# For this, we need to ensure the latch is set from Scenario 3.
# We can run scenario 3, then scenario 4 using the *same network object*
# so the N_Latch state persists.
print("\nScenario 4: N1_Prime alone (AFTER Latch is set) - N_Target SHOULD fire")
# Assuming net_Th7 still has N_Latch=1 from the end of Scenario 3.
# If running independently, you'd need an input sequence that first sets the latch.
# Let's make an input sequence that does both:
inputs_s4_combined = [
    {"N1_Prime": 1, "N2_Pairing": 1}, # t=0: Pairing, N_Target=0 initially.
                                      # N_AND_Inputs fires at t=1. N_Target fires at t=1 (from N2_Pairing).
    {"N1_Prime": 0, "N2_Pairing": 0}, # t=1: N_Latch fires at t=2 (from N_AND_Inputs(1)). N_Target=0.
    {"N1_Prime": 1, "N2_Pairing": 0}, # t=2: N_Latch is ON at t=2. N1_Prime is ON at t=2.
                                      # N_AND_PrimeLatch fires at t=3. N_Target fires at t=3.
    {"N1_Prime": 0, "N2_Pairing": 0}, # t=3
    None                              # t=4
]
net_Th7.run(time_steps=5, input_sequence=inputs_s4_combined)
net_Th7.print_history(neuron_ids=neuron_order_Th7)


--- Theorem 7: Alterable Synapses by Circles (Fig 1i logic) ---

Scenario 1: N1_Prime alone (before pairing) - N_Target should NOT fire from N1_Prime
Time | N1_Prime | N2_Pairing | N_AND_Inputs | N_Latch | N_AND_PrimeLatch | N_Target
-----------------------------------------------------------------------------------
0    |    1    |    0    |    0    |    0    |    0    |    0   
1    |    0    |    0    |    0    |    0    |    0    |    0   
2    |    0    |    0    |    0    |    0    |    0    |    0   
3    |    0    |    0    |    0    |    0    |    0    |    0   

Scenario 2: N2_Pairing alone - N_Target SHOULD fire (direct path)
Time | N1_Prime | N2_Pairing | N_AND_Inputs | N_Latch | N_AND_PrimeLatch | N_Target
-----------------------------------------------------------------------------------
0    |    0    |    1    |    0    |    0    |    0    |    0   
1    |    0    |    0    |    0    |    0    |    0    |    1   
2    |    0    |    0    |    0    |    0    |    0    |